# DS 340W Group 28: Interpretation-First Text-to-SQL (AmbiQT): Reproducibility Demo

**Sanjana Menon & Sharvari Purighalla**

This notebook reproduces every result in our paper on a **fresh Google Colab runtime**. It
downloads the code from GitHub, installs everything, runs Agents **A / A2 / B** on the 12-question
AmbiQT pilot with **Qwen3:4B** (`num_predict=8192`), and prints the results table from the paper.

**Before you start:** `Runtime → Change runtime type → T4 GPU`. Then run the cells **top to bottom, one at a time**.
No file uploads and no Google Drive access are needed.

| Step | Time on a T4 |
|---|---|
| 1–3 Clone, install, offline checks | ~1 min |
| 4–5 Install Ollama, pull Qwen3:4B, smoke test | ~3–5 min |
| 6 Full 36-request run | **~40 min** |
| 7–8 Results tables | seconds |

## 1. Confirm the GPU

In [ ]:
!nvidia-smi

## 2. Download our code from GitHub

In [ ]:
%cd /content
!rm -rf ds340w-finalproject
!git clone https://github.com/spp6039/ds340w-finalproject.git
%cd /content/ds340w-finalproject/colab_pilot
!git log -1 --format='Commit: %h  %s'
!ls

## 3. Install Python dependencies and run the offline checks (no model)
`prepare_and_verify.py` checks that all 12 benchmark records, SQLite databases and gold queries load,
and that the agents **cannot** see gold answers. The unit tests check the orchestration with a mocked model.
Expect `RESULT: ALL CHECKS PASS` and every test suite to `PASS`.

In [ ]:
!pip -q install -r requirements.txt
!python prepare_and_verify.py
!python test_orchestration.py | tail -1
!python test_run_pilot_config.py | tail -1

## 4. Install Ollama, start the server, and pull `qwen3:4b`
We pin the Ollama version that produced our paper results. The model digest printed by
`ollama list` should be **`359d7dd4bcda`**.

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION=0.40.18 sh 2>&1 | tail -3

In [ ]:
import subprocess, time, requests
proc = subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama_server.log', 'a'),
                        stderr=subprocess.STDOUT, start_new_session=True)
for _ in range(60):
    try:
        if requests.get('http://127.0.0.1:11434/api/tags', timeout=3).ok:
            print('Ollama server ready'); break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise RuntimeError('Ollama did not start; see /content/ollama_server.log')

In [ ]:
!ollama pull qwen3:4b > /dev/null 2>&1
!ollama --version
!ollama list

## 5. Smoke test: one real request to the model
Expect `RESULT: SMOKE TEST PASS`. The first call also warms the model onto the GPU.

In [ ]:
!python smoke_test.py --smoke-timeout 180

## 6. Run the full experiment: 12 questions × 3 agents = 36 requests
Settings shared by all three agents: `qwen3:4b`, temperature 0.0, top_p 1.0, `num_predict=8192`,
`think=True`, 600 s request timeout, one LLM call per question. Each line printed is one
(question, agent) request. Expect the run to end with `RUN OK: 36/36 requests written, 0 request_error`.

In [ ]:
!python -u run_pilot.py --out /content/demo_results --num-predict 8192

## 7. Results of THIS run, compared with the committed paper run
`summarize_results.py` reads only the saved JSONL. The comparison checks each of the 36
(agent, question) outcomes against `results/pilot_generations_20261124T194002Z.jsonl`, the run reported in the paper.

In [ ]:
import glob
new_run = sorted(glob.glob('/content/demo_results/pilot_generations_*.jsonl'))[-1]
print('New run file:', new_run)
!python summarize_results.py "{new_run}" --reference ../results/pilot_generations_20261124T194002Z.jsonl

## 8. Paper tables from the committed result files
These are the two runs used in the paper: the **4,096-token** preliminary budget run and the
**8,192-token** final run. They need no GPU, so this cell can be run on its own as well.

In [ ]:
!python summarize_results.py \
    ../results/pilot_generations_20261119T191703Z.jsonl \
    ../results/pilot_generations_20261124T194002Z.jsonl

### The 8,192-token results reported in the paper
| Agent | Completed | Any-alt coverage | Both-alt coverage | Alt-interp recall | Tokens |
|---|---|---|---|---|---|
| A (single) | 12/12 | 8/12 | 0/12 | 8/24 (33%) | 28,425 |
| A2 (multi, K=2) | 12/12 | 8/12 | 3/12 | 11/24 (46%) | 39,188 |
| B (interpretation-first, K=2) | 11/12 (1 parse error) | 7/12 | 4/12 | 11/24 (46%) | 47,070 |